# 04 - Transformer with Defense 2 (Output Limiting)

Defense: temperature scaling or label-only outputs

In [1]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
DEFENSE_MODE = 'temperature'  # temperature | label_only
TEMPERATURE = 4.0

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def build_transformer(n_features, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def limit_output(p):
    p = np.clip(np.asarray(p), 1e-6, 1-1e-6)
    if DEFENSE_MODE == 'label_only':
        return (p >= 0.5).astype(float)
    logit = np.log(p/(1-p))
    q = 1.0/(1.0+np.exp(-logit/max(TEMPERATURE,1e-6)))
    return np.clip(q, 1e-6, 1-1e-6)

def mia_features(p, y):
    p = np.clip(np.asarray(p), 1e-8, 1-1e-8)
    y = np.asarray(y)
    loss = -(y*np.log(p) + (1-y)*np.log(1-p))
    conf = np.maximum(p,1-p)
    ent = -(p*np.log(p)+(1-p)*np.log(1-p))
    cor = ((p>=0.5).astype(int)==y).astype(float)
    margin = np.abs(p-0.5)
    return np.column_stack([loss,conf,ent,cor,margin])

def attack_row(name, y_true, y_pred, y_score):
    return {'attack':name,'auc':roc_auc_score(y_true,y_score),'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0)}

def run_attacks(model, Xtr_seq, ytr, Xte_seq, yte, Xall, yall, seed, transform_fn):
    p_tr = transform_fn(model.predict(Xtr_seq, verbose=0).ravel())
    p_te = transform_fn(model.predict(Xte_seq, verbose=0).ravel())
    Fm, Fn = mia_features(p_tr, ytr), mia_features(p_te, yte)
    Xa = np.vstack([Fm, Fn]); ya = np.concatenate([np.ones(len(Fm),dtype=int), np.zeros(len(Fn),dtype=int)])
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.4, random_state=seed, stratify=ya)

    score_tr = -Xa_tr[:,0]; score_te = -Xa_te[:,0]
    thrs = np.linspace(score_tr.min(), score_tr.max(), 300)
    best_t, best_b = thrs[0], -1
    for t in thrs:
        pred = (score_tr >= t).astype(int)
        tpr = ((pred==1)&(ya_tr==1)).sum()/max((ya_tr==1).sum(),1)
        tnr = ((pred==0)&(ya_tr==0)).sum()/max((ya_tr==0).sum(),1)
        b = 0.5*(tpr+tnr)
        if b > best_b: best_b, best_t = b, t
    r_thr = attack_row('threshold_loss', ya_te, (score_te>=best_t).astype(int), score_te)

    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(Xa_tr, ya_tr)
    plr = lr.predict_proba(Xa_te)[:,1]
    r_lr = attack_row('logistic', ya_te, (plr>=0.5).astype(int), plr)

    Xab, Xaux, yab, yaux = train_test_split(Xall, yall, test_size=0.30, random_state=seed, stratify=yall)
    Xm, Xn, ym, yn = train_test_split(Xab, yab, test_size=0.45, random_state=seed+1, stratify=yab)
    sct = StandardScaler(); Xm_sc = sct.fit_transform(Xm).astype(np.float32); Xn_sc = sct.transform(Xn).astype(np.float32)
    Xm_seq = Xm_sc.reshape(-1,1,Xm_sc.shape[1]); Xn_seq = Xn_sc.reshape(-1,1,Xn_sc.shape[1])
    set_seed(seed+1000); tf.keras.backend.clear_session()
    tgt = build_transformer(Xm_sc.shape[1], dropout=0.15); tgt.fit(Xm_seq, ym, epochs=80, batch_size=16, verbose=0)
    ptm = transform_fn(tgt.predict(Xm_seq, verbose=0).ravel()); ptn = transform_fn(tgt.predict(Xn_seq, verbose=0).ravel())
    Xt = np.vstack([mia_features(ptm, ym), mia_features(ptn, yn)])
    yt = np.concatenate([np.ones(len(ym),dtype=int), np.zeros(len(yn),dtype=int)])

    SX, SY = [], []
    for i in range(10):
        xsm, xsn, ysm, ysn = train_test_split(Xaux, yaux, test_size=0.5, random_state=seed+100+i, stratify=yaux)
        scs = StandardScaler(); xsm_sc = scs.fit_transform(xsm).astype(np.float32); xsn_sc = scs.transform(xsn).astype(np.float32)
        xsm_seq = xsm_sc.reshape(-1,1,xsm_sc.shape[1]); xsn_seq = xsn_sc.reshape(-1,1,xsn_sc.shape[1])
        set_seed(seed+2000+i); tf.keras.backend.clear_session()
        sh = build_transformer(xsm_sc.shape[1], dropout=0.15); sh.fit(xsm_seq, ysm, epochs=40, batch_size=16, verbose=0)
        psm = transform_fn(sh.predict(xsm_seq, verbose=0).ravel()); psn = transform_fn(sh.predict(xsn_seq, verbose=0).ravel())
        SX.append(np.vstack([mia_features(psm, ysm), mia_features(psn, ysn)]))
        SY.append(np.concatenate([np.ones(len(ysm),dtype=int), np.zeros(len(ysn),dtype=int)]))

    meta = GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=3, random_state=seed)
    meta.fit(np.vstack(SX), np.concatenate(SY))
    sc = meta.predict_proba(Xt)[:,1]
    r_sh = attack_row('shadow_meta', yt, (sc>=0.5).astype(int), sc)
    return pd.DataFrame([r_thr, r_lr, r_sh]).sort_values('auc', ascending=False)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9)


In [4]:
print('=== Train baseline model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model = build_transformer(X_train.shape[1], dropout=0.15)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
h = model.fit(X_train_seq, y_train, epochs=120, batch_size=32, validation_split=0.2, callbacks=[es], verbose=0)
p_te = model.predict(X_test_seq, verbose=0).ravel()
test_acc = accuracy_score(y_test, (p_te>=0.5).astype(int))
print(f'test_acc={test_acc:.4f}, epochs={len(h.history["loss"])}')

=== Train baseline model ===
test_acc=0.9331, epochs=18


In [5]:
print('=== Attacks baseline outputs ===')
attack_base = run_attacks(model, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED, transform_fn=lambda p: p)
display(attack_base.round(4))

=== Attacks baseline outputs ===


,attack,auc,accuracy,precision,recall,f1
2,shadow_meta,0.5395,0.5668,0.5686,0.8593,0.6844
0,threshold_loss,0.5354,0.4930,0.2317,0.6786,0.3455
1,logistic,0.5276,0.8028,0.0000,0.0000,0.0000


In [6]:
print(f'=== Attacks with Defense 2 ({DEFENSE_MODE}) ===')
attack_def2 = run_attacks(model, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED+7, transform_fn=limit_output)
display(attack_def2.round(4))

=== Attacks with Defense 2 (temperature) ===


,attack,auc,accuracy,precision,recall,f1
2,shadow_meta,0.5231,0.4696,0.5303,0.2593,0.3483
0,threshold_loss,0.5028,0.4507,0.2159,0.6786,0.3276
1,logistic,0.5028,0.8028,0.0000,0.0000,0.0000


In [7]:
print('=== Baseline vs Defense 2 ===')
cmp = attack_base[['attack','auc','accuracy','f1']].merge(attack_def2[['attack','auc','accuracy','f1']], on='attack', suffixes=('_baseline','_defense2'))
cmp['delta_auc'] = cmp['auc_defense2'] - cmp['auc_baseline']
cmp['delta_accuracy'] = cmp['accuracy_defense2'] - cmp['accuracy_baseline']
cmp['delta_f1'] = cmp['f1_defense2'] - cmp['f1_baseline']
display(cmp.sort_values('attack').round(4))

=== Baseline vs Defense 2 ===


,attack,auc_baseline,accuracy_baseline,f1_baseline,auc_defense2,accuracy_defense2,f1_defense2,delta_auc,delta_accuracy,delta_f1
2,logistic,0.5276,0.8028,0.0000,0.5028,0.8028,0.0000,-0.0247,0.0000,0.0000
0,shadow_meta,0.5395,0.5668,0.6844,0.5231,0.4696,0.3483,-0.0163,-0.0972,-0.3361
1,threshold_loss,0.5354,0.4930,0.3455,0.5028,0.4507,0.3276,-0.0326,-0.0423,-0.0179


In [8]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack','auc_baseline','auc_defense2','delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_baseline,auc_defense2,delta_auc
1,threshold_loss,0.5354,0.5028,-0.0326
2,logistic,0.5276,0.5028,-0.0247
0,shadow_meta,0.5395,0.5231,-0.0163
